# EDA - Point Reyes Mushrooms

**Objective:** Explore available data and examine relationships between land cover, weather and mushroom fruitings.

**Key Questions:**
1. What environments facilitate porcini mushroom growth?
2. What weather conditions lead to fruiting?

**Project Plan**
Two models:
1. Spatial - Two options
    - A. Simple yes/no - is there Bishop Pine forest (the well-known best host for porcini in Point Reyes)
        - We will use this approach for now, given it is simple and common knowledge
    - B. Model relative suitability of environments - Porcini can also grow with Douglas Fir, for example. 
        - Keep in mind for future work. A model that can predict suitable environments will allow us to generalize to other mushroom species.
2. Temporal
    - Use lagging indicators of precipitation and temperature to predict mushroom emergence.
    - Will need to adjust for observer effort (more reports on weekends, nice weather days) - strategy: include day of week, current weather in training data, predict using an "ideal day" (e.g. Saturday, sunny, 15C)

## **Setup**

In [ ]:
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from scipy.spatial.distance import cdist
import numpy as np
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as cx
import folium
# import os

# --- Configuration ---
pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")


In [ ]:
# Get Point Reyes boundaries
# ================================================
cpad = gpd.read_file("../Data/CPAD_Release_2025b/CPAD_2025b_Units/CPAD_2025b_Units.shp")
pt_reyes = cpad[cpad['UNIT_NAME'] == 'Point Reyes National Seashore'].copy()
pt_reyes_boundary = pt_reyes.dissolve()
# Save
# pt_reyes_boundary.to_file("point_reyes_boundary.geojson", driver='GeoJSON')

# Plot
# ================================================
# Reproject to Web Mercator (EPSG:3857)
pt_reyes_web_mercator = pt_reyes_boundary.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 10))
pt_reyes_web_mercator.plot(ax=ax, alpha=1.0, facecolor='none', edgecolor='r', linewidth=2)
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

# **Spatial Component - Land Cover Data**

Identify regions with host trees for porcini

### CALVEG Land Cover Data - DEPRECATED
***Marin Fine Grain Survey Data is more accurate (see below).** The following three cells are left for illustrative purposes.

In [ ]:
print("Loading vegetation data...")
gdb_layers = [
    {'path': '../Data/land_cover_data_CALVEG/S_USA.EVMid_R05_CentralCoast.gdb', 'layer': 'EVMid_R05_CentralCoast'},
    {'path': '../Data/land_cover_data_CALVEG/S_USA.EVMid_R05_NorCoastMid.gdb', 'layer': 'EVMid_R05_NorCoastMid'},
    {'path': '../Data/land_cover_data_CALVEG/S_USA.EVMid_R05_NorCoastWest.gdb', 'layer': 'EVMid_R05_NorCoastWest'}
]
veg_gdfs = [gpd.read_file(item['path'], layer=item['layer']) for item in gdb_layers]
veg_gdf = pd.concat(veg_gdfs, ignore_index=True)

# # Print available veg_gdf columns in alphabetical order
# print("Available vegetation columns:")
# for col in sorted(veg_gdf.columns.tolist()):
#     print(col)

# REGIONAL_DOMINANCE_TYPE gives the dominant tree species in the area. 
# Porcini grow primarilyin Bishop Pine and Douglas-fir forests.
# PM = Bishop Pine (Primary)
# DF = Douglas-fir (Secondary)
# QA = Coast Live Oak (Occasional)
# QT = Tan Oak (Occasional)
# KP = Knobcone Pine (Occasional)
# (REGIONAL_DOMINANCE_TYPE_2 and REGIONAL_DOMINANCE_TYPE_3 do not contain any PM, DF codes)
print(veg_gdf.REGIONAL_DOMINANCE_TYPE.value_counts().loc[['PM', 'DF', 'QA', 'QT', 'KP']])

# Other columns of potential interest:
print(veg_gdf.CONIFER_CFA_CLASS.value_counts()) # Conifer Cover From Above - measure of density (01: <10%, 20: 10-29.9%, 40: 30-59.9%, 80: 60-100%)
print(veg_gdf.CON_CFA.value_counts()) # Conifer Cover From Above - numbers represent midpoint of percentage cover (10% bands)
print(veg_gdf.COVERTYPE.value_counts()) # Cover type (conifer, hardwood, mixed, herbaceous, shrub, urban, etc)
print(veg_gdf.OS_TREE_DIAMETER_CLASS.value_counts())


In [ ]:
import folium

# Define the specific CALVEG codes for Point Reyes hosts
host_codes = ['PM', 'DF', 'QA', 'TO', 'KP']

# # Define the density classes to keep (Excluding '01' - Trace)
# # 20=Sparse, 40=Moderate, 80=Dense
# viable_density = ['20', '40', '80']

# Filter the original CALVEG dataset (assuming 'calveg' is loaded)
# We filter for Species matches AND Density matches
potential_habitat = veg_gdf[
    (veg_gdf['REGIONAL_DOMINANCE_TYPE'].isin(host_codes)) # & (calveg['CONIFER_CFA_CLASS'].isin(viable_density))
]

# Clip to the Park Boundary (assuming 'pt_reyes_boundary' is loaded)
# Ensure CRS match first
if potential_habitat.crs != pt_reyes_boundary.crs:
    potential_habitat = potential_habitat.to_crs(pt_reyes_boundary.crs)

pt_reyes_habitat = gpd.clip(potential_habitat, pt_reyes_boundary)




# 1. Reproject to Lat/Lon (EPSG:4326)
# Folium strictly requires Latitude/Longitude
boundary_ll = pt_reyes_boundary.to_crs(epsg=4326)
habitat_ll = pt_reyes_habitat.to_crs(epsg=4326)

# Convert any datetime/timestamp columns to strings for JSON serialization
# Folium cannot serialize Timestamp objects, so we need to convert them
for col in habitat_ll.columns:
    if col == 'geometry':
        continue  # Skip geometry column
    if pd.api.types.is_datetime64_any_dtype(habitat_ll[col]):
        habitat_ll[col] = habitat_ll[col].astype(str)
    elif habitat_ll[col].dtype == 'object':
        # Check if object column contains Timestamp objects
        sample = habitat_ll[col].dropna()
        if len(sample) > 0 and isinstance(sample.iloc[0], pd.Timestamp):
            habitat_ll[col] = habitat_ll[col].astype(str)

# 2. Initialize the Map
# Center it on Point Reyes automatically
center = boundary_ll.geometry.centroid.iloc[0]
m = folium.Map(location=[center.y, center.x], zoom_start=11, tiles=None)

folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Satellite (Esri)',
    overlay=False,  # This is a background layer
    control=True
).add_to(m)

# 3. Define Colors for Species (Optional but helpful)
# Maps your specific codes to colors
color_map = {
    'PM': 'red',     # Bishop Pine
    'DF': 'orange',  # Douglas Fir
    'QA': 'yellow',  # Oak
    'QT': 'yellow',  # Tanoak
    'KP': 'yellow'   # Knobcone Pine
}

def style_function(feature):
    """Function to color habitat polygons by species code"""
    code = feature['properties']['REGIONAL_DOMINANCE_TYPE']
    return {
        'fillColor': color_map.get(code, 'gray'),
        'color': 'black',    # Border of the habitat polygons
        'weight': 0.5,       # Border line weight
        'fillOpacity': 0.6
    }

# 4. Add the Habitat Layer
folium.GeoJson(
    habitat_ll,
    name="Porcini Habitat",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['COVERTYPE', 'REGIONAL_DOMINANCE_TYPE'])
).add_to(m)

# 5. Add the Point Reyes Boundary (Red Outline)
folium.GeoJson(
    boundary_ll,
    name="Point Reyes Boundary",
    style_function=lambda x: {
        'color': 'red',
        'weight': 2,
        'fillOpacity': 0
    }
).add_to(m)

# 6. Add Layer Control and Display
folium.LayerControl().add_to(m)
m

In [ ]:
# If Folium plot is blocked by Jupyter notebook, save map to html and open with a web browser
import webbrowser
import os

# Save the map
output_file = "porcini_map.html"
m.save(output_file)

# Open it automatically in browser
webbrowser.open('file://' + os.path.realpath(output_file))

### Marin County Fine Grain Land Cover Data

The CALVEG map has some obvious inaccuracies! Forest boundaries do not always line up, and there are some substantial stands of Bishop Pine that are entirely omitted.

There is another land cover dataset of Point Reyes from 2018, when Marin County used aerial surveying to obtain a high resolution dataset of vegetation. Let's look at the Marin County Fine Scale Vegetation Map to see if it gives a better representation of Bishop Pine and Douglas Fir distribution.

#### Investigate dataset columns

**Relevant columns**
- MAP_CLASS_18: Classification (typically dominant species), e.g. "Pinus muricata – Pinus radiata Alliance" which is Bishop Pine
- ABBRV: Classification abbreviation, e.g. "PiMu" for "Pinus muricata – Pinus radiata Alliance"
- CON_COVER_18: Percent conifer cover
- 'STAND_HT_MN_19', 'STAND_HT_MX_19', 'STAND_HT_SD_19'
- WOODWARD_FIRE_SEVERITY: Burn severity + percent of parcel, e.g. "Low - 68.8%, Medium - 31.2%"

In [ ]:
veg_map_path = "../Data/land_cover_data_MC_fine_scale/Finescale_Veg_9_30_21.gdb" 
MC_finescale_gdf = gpd.read_file(veg_map_path, layer='MARIN_FINESCALE_VEG_9_30_2021')

print(MC_finescale_gdf.columns)
print('\n')
# print(MC_finescale_gdf.head())

# Value counts of potentially useful columns
# ---------------------------------------------
pd.set_option('display.max_rows', 50) # set to None to see all values

# Classification (gives dominant species)
# print(MC_finescale_gdf.value_counts(subset=["ABBRV", "MAP_CLASS_18"]))
# Filter for rows containing "Pinus muricata" or "Pseudotsuga menziesii" (case-insensitive)
filtered_gdf = MC_finescale_gdf[
    MC_finescale_gdf["MAP_CLASS_18"].str.contains("Pinus muricata|Pseudotsuga menziesii", case=False, na=False)
]
print(filtered_gdf.value_counts(subset=["ABBRV", "MAP_CLASS_18"]))
# Bishop Pine:
#     PiMu, Pinus muricata – Pinus radiata Alliance
# Douglas Fir:
#     PsMe, Pseudotsuga menziesii – (Notholithocarpus densiflorus – Arbutus menziesii) Alliance
# Coast Live Oak:
#     QuAg, Quercus agrifolia Alliance
# Tanoak:
#     NoDe, Notholithocarpus densiflorus Alliance

# Conifer cover (%)
print(MC_finescale_gdf.CON_COVER_18.value_counts())
print('\n')

# Woodward fire severity
print(MC_finescale_gdf.WOODWARD_FIRE_SEVERITY.value_counts())
print('\n')
# Gives percentage of area burned and classification of severity: Low, Medium, High, Very High, Outside of Woodward Fire
# Can be a mix of classifications, e.g. "Low - 68.8%, Medium - 31.2%"

# Higher level classification of land cover, e.g. "Evergreen Hardwood", "Native Shrub", "Conifer", "Tidal Wetland", "Developed" 
# Not likely to need - ABBRV/MAP_CLASS_18 provide better detail
print(MC_finescale_gdf.FOREST_LIFEFORM_18.value_counts())
print('\n')



#### Interactive map of Marin County Fine Grain data

In [ ]:

# ---------------------------------------------------------
# 1. SETUP & DEFINITIONS
# ---------------------------------------------------------

# Mapping the specific ABBRV codes to readable names and colors
species_config = {
    'PiMu': {'name': 'Bishop Pine', 'color': '#228B22'},   # Forest Green
    'PsMe': {'name': 'Douglas Fir', 'color': '#00008B'},   # Dark Blue
    'QuAg': {'name': 'Coast Live Oak', 'color': '#FF8C00'},# Dark Orange
    'NoDe': {'name': 'Tanoak', 'color': '#800080'}        # Purple
}

# ---------------------------------------------------------
# 2. PROCESS VEGETATION LAYER (Habitat)
# ---------------------------------------------------------

# Filter for only the species we care about
# We use .copy() to avoid SettingWithCopy warnings
veg_gdf = MC_finescale_gdf[MC_finescale_gdf['ABBRV'].isin(species_config.keys())].copy()

# Dissolve by species to merge thousands of small adjacent parcels into clean blocks
# This significantly speeds up map rendering
veg_dissolved = veg_gdf.dissolve(by='ABBRV', as_index=False)
# Save the dissolved vegetation layer to a GeoPackage (recommended for spatial data)
veg_dissolved.to_file("marin_finescale_veg_dissolved.gpkg", layer='veg_dissolved', driver="GPKG")


# ---------------------------------------------------------
# 3. PROCESS FIRE LAYER (Burn Severity)
# ---------------------------------------------------------

def categorize_burn(severity_str):
    """
    Parses the complex severity string (e.g., "High - 77.9%, Very High - 16.6%")
    into a simple category for plotting.
    """
    if not isinstance(severity_str, str) or severity_str.startswith('Outside'):
        return None
    
    # Prioritize the most severe mention
    if 'Very High' in severity_str:
        return 'Very High'
    elif 'High' in severity_str:
        return 'High'
    elif 'Medium' in severity_str:
        return 'Medium'
    elif 'Low' in severity_str:
        return 'Low'
    return None

# Apply categorization
fire_gdf = MC_finescale_gdf.copy() 
fire_gdf['burn_category'] = fire_gdf['WOODWARD_FIRE_SEVERITY'].apply(categorize_burn)

# Filter out unburned areas
fire_gdf = fire_gdf.dropna(subset=['burn_category'])

# # Dissolve by burn category to create clean "Zones of Destruction"
# fire_dissolved = fire_gdf.dissolve(by='burn_category', as_index=False)
# Dissolve all areas (for a cleaner map with one simple red outline denoting burned areas)
fire_fully_dissolved = fire_gdf.dissolve()

# # Define burn colors (Red scale)
# # ** When layering with vegetaion coloring, use one red color for better readability
# # For now, I only color burn severity of medium or greater
# burn_colors = {
#     'Very High':'#FF0000', # Red     '#8B0000', # Dark Red
#     'High':     '#FF0000', # Red
#     'Medium':   '#FF0000', # Red     '#CD5C5C', # Indian Red
#     'Low':      None,      #         '#FFA07A'  # Light Salmon
# }

# ---------------------------------------------------------
# 4. PLOTTING (Folium)
# ---------------------------------------------------------

# Center map on Point Reyes (approximate)
m = folium.Map(location=[38.04, -122.85], zoom_start=12, tiles=None)

# A. Add Satellite Base Map (Esri World Imagery)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Satellite (Esri)',
    overlay=False,
    control=True
).add_to(m)

# B. Add Vegetation Layer
# We iterate through species to add them as separate groups (toggleable)
for code, props in species_config.items():
    subset = veg_dissolved[veg_dissolved['ABBRV'] == code]
    
    if not subset.empty:
        folium.GeoJson(
            subset,
            name=f"Habitat: {props['name']}",
            style_function=lambda x, color=props['color']: {
                'fillColor': color,
                'color': color,
                'weight': 1,
                'fillOpacity': 0.6
            },
            tooltip=f"{props['name']} (Habitat)"
        ).add_to(m)

# C. Add Fire Severity Layer
# folium.GeoJson(
#     fire_dissolved,
#     name="Woodward Fire Scars",
#     style_function=lambda x: {
#         'fillColor': None, #burn_colors.get(x['properties']['burn_category'], 'gray'),
#         'color': burn_colors.get(x['properties']['burn_category'], 'gray'),
#         'weight': 0.5,
#         'fillOpacity': 0.0  # Semi-transparent so you can see the trees underneath
#     tooltip=folium.GeoJsonTooltip(fields=['burn_category'], aliases=['Burn Severity:'])
# ).add_to(m)
folium.GeoJson(
    fire_fully_dissolved,
    name="Woodward Fire Scars",
    style_function=lambda x: {
        'fillColor': None,
        'color': 'red',
        'weight': 1.5,
        'fillOpacity': 0.0
    },
    # Add as a non-interactive layer so it doesn't capture mouse events
    highlight_function=None,
    tooltip=None,
    interactive=False
).add_to(m)

# D. Add Layer Control
folium.LayerControl().add_to(m)

# Display
m

In [ ]:
# Save map as html and open in browser, if not showing properly in notebook (e.g. no base map, or notebook not trusted)
import webbrowser
import os

# Save the map
output_file = "marin_fine_grain_land_cover_map.html"
m.save(output_file)

# Open it automatically in browser
webbrowser.open('file://' + os.path.realpath(output_file))

### MVP Spatial Model - Bishop Pine is best

For the first iteration of the spatial component, we can use our human knowledge (without any machine learning model). We will consider any Bishop Pine forest that has not been affected by fire as prime porcini territory. Unburned Douglas Fir forest will be considered secondary territory.

Future improvements we can explore include:
1) Account for other forest characteristics (CON_COVER_18, 'STAND_HT_MN_19', 'STAND_HT_MX_19', 'STAND_HT_SD_19')
    - Consider areas with greater conifer conver / stand height (stand height relates to forest maturity - porcini mycelium can take 10 years to fruit) to be better habitat
2) Apply a machine learning model
    - Training a model on all relevant variables could give better estimates of the relative productivity of different environments and could help identify the best places to find porcini
3) A full, combined spatial/temporal model
    - What if the timing of porcini fruiting varies between different environments? A model trained on both spatial and temporal features could capture those differences.

In [ ]:
# Limit to just Bishop Pine (best host) and Douglas Fir (secondary host)
species_config = {
    'PiMu': {'name': 'Bishop Pine', 'color': '#228B22'},   # Forest Green
    'PsMe': {'name': 'Douglas Fir', 'color': '#00008B'},   # Dark Blue
    # 'QuAg': {'name': 'Coast Live Oak', 'color': '#FF8C00'},# Dark Orange
    # 'NoDe': {'name': 'Tanoak', 'color': '#800080'}        # Purple
}

# Define map boundaries
boundary_ll = pt_reyes_boundary.to_crs(epsg=4326)
minx, miny, maxx, maxy = boundary_ll.total_bounds

buffer = 0.05
corner_sw = [miny - buffer, minx - buffer]
corner_ne = [maxy + buffer, maxx + buffer]
hard_bounds = [corner_sw, corner_ne]

# Initialize map
m = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=11,
    tiles=None,
    max_bounds=hard_bounds,  
    max_bounds_viscosity=1.0,  # 1.0 = Solid Wall, 0.0 = Rubber Band
    min_zoom=11, # (10 is roughly "County View")
    # Disable "World Copying" (prevents scrolling endlessly left/right)
    no_wrap=True
)

# Add trail map (useful for planning foraging hikes)
folium.TileLayer(
    tiles='https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png',
    attr='Map data: &copy; OpenStreetMap contributors, SRTM | Map style: &copy; OpenTopoMap (CC-BY-SA)',
    name='Hiking Map (OpenTopoMap)',
    overlay=False,
    control=True
).add_to(m)

# Add Point Reyes Boundary
folium.GeoJson(
    boundary_ll,
    name="Point Reyes Boundary",
    style_function=lambda x: {
        'fillColor': 'none',
        'color': 'black',
        'weight': 4,
        'fillOpacity': 0
    },
    tooltip="Point Reyes National Seashore"
).add_to(m)

# Add a partially transparent grey overlay to areas outside of the Pt Reyes boundary
# Create a rectangle covering the whole visible map, then cut the Pt Reyes boundary out of it to shade only outside
from shapely.geometry import box
import geopandas as gpd

# Create a big rectangle covering the map extent (`buffer` already above is 0.05 deg)
mask_rect = box(minx - buffer, miny - buffer, maxx + buffer, maxy + buffer)
# Subtract the boundary polygon from the rectangle
mask_gdf = gpd.GeoDataFrame(geometry=[mask_rect], crs=boundary_ll.crs)
# Apply difference to get the "outside" ring
outside_geom = mask_rect.difference(boundary_ll.unary_union)
outside_gdf = gpd.GeoDataFrame(geometry=[outside_geom], crs=boundary_ll.crs)

# Add to folium
folium.GeoJson(
    outside_gdf,
    name="Outside Pt Reyes (Grey Mask)",
    style_function=lambda x: {
        'fillColor': 'grey',
        # 'color': 'grey',
        'weight': 0,
        'fillOpacity': 0.8
    }
).add_to(m)


# Add Vegetation Layer (With slightly increased transparency)
# Since the basemap has details (contour lines), we make the trees a bit more transparent (0.5)
for code, props in species_config.items():
    subset = veg_dissolved[veg_dissolved['ABBRV'] == code]
    
    if not subset.empty:
        folium.GeoJson(
            subset,
            name=f"Habitat: {props['name']}",
            style_function=lambda x, color=props['color']: {
                'fillColor': color,
                'color': color,       # Keep outline matching fill
                'weight': 1,
                'fillOpacity': 0.5    # More see-through to see the trails underneath
            },
            tooltip=f"{props['name']} (Habitat)"
        ).add_to(m)

# Add Fire Layer (transparent, non-interactive)
folium.GeoJson(
    fire_fully_dissolved,
    name="Woodward Fire Scars",
    style_function=lambda x: {
        'fillColor': 'red',
        'color': 'red',
        'weight': 1.5,
        'fillOpacity': 0.1
    },
    # Add as a non-interactive layer so it doesn't capture mouse events
    highlight_function=None,
    tooltip=None,
    interactive=False
).add_to(m)

# 5. Layer Control
folium.LayerControl().add_to(m)

# Add a legend for the vegetation habitat colors, and include Point Reyes boundary and burned area boundary

legend_html = '''
 <div style="
     position: fixed;
     bottom: 50px;
     left: 50px;
     z-index: 9999;
     background: white;
     padding: 10px;
     border: 2px solid grey;
     border-radius: 6px;
     box-shadow: 0 2px 6px rgba(0,0,0,0.2);
     font-size: 14px;
     ">
 <b>Legend</b><br>
 {}
 </div>'''

legend_items = ""

# Add Pt Reyes boundary
legend_items += '''
    <div style="margin-bottom:3px;">
        <span style="display:inline-block;width:18px;height:18px;background:none;border:3px solid black;margin-right:6px;"></span>
        <span style="vertical-align:middle;">Pt Reyes Natl. Seashore Boundary</span>
    </div>
'''

# Add Woodward Fire boundary
legend_items += '''
    <div style="margin-bottom:3px;">
        <span style="display:inline-block;width:18px;height:18px;background:none;border:2px solid red;margin-right:6px;"></span>
        <span style="vertical-align:middle;">Woodward Fire (2020)</span>
    </div>
'''

# Add vegetation habitats
for code, props in species_config.items():
    legend_items += f'''<div style="margin-bottom:3px;">
        <span style="display:inline-block;width:18px;height:18px;background:{props['color']};border:1px solid #999;margin-right:6px;"></span>
        {props['name']}
    </div>'''

m.get_root().html.add_child(folium.Element(legend_html.format(legend_items)))


m.fit_bounds(hard_bounds)
m


In [ ]:
# Save map as html and open in browser, if not showing in notebook (or for a larger map)
import webbrowser
import os

# Save the map
output_file = "vegetation_trail_map.html"
m.save(output_file)

# Open it automatically in browser
webbrowser.open('file://' + os.path.realpath(output_file))

In [ ]:
# ** Come back to this if decide to move to React webapp **

# # Save map data to .geojson files for React webapp

# import os

# # 1. Create a generic output folder (optional)
# os.makedirs("web_data", exist_ok=True)

# # 2. Export Vegetation Layer
# # We strictly convert to EPSG:4326 (Lat/Lon) because web maps require it.
# # We also round coordinates to 6 decimal places to keep file size small (faster loading).
# veg_dissolved.to_crs(epsg=4326).to_file(
#     "cleaned_habitat.geojson", 
#     driver='GeoJSON', 
#     engine='pyogrio'  # Faster if installed, otherwise remove this argument
# )

# # 3. Export Fire Layer
# fire_dissolved.to_crs(epsg=4326).to_file(
#     "woodward_fire.geojson", 
#     driver='GeoJSON'
# )

# # print("Files saved to 'web_data/' folder!")

# **Temporal Component - Weather conditions and mushroom emergence**

### Weather station data

Many stations have sparse or missing data. We will use reanalysis data instead (see next section).

In [ ]:
# weather_files = glob.glob('../Data/weather_data_stations/bay_area_weather_data_*.csv')
# weather_df = pd.concat([pd.read_csv(f) for f in weather_files], ignore_index=True)
# weather_df['date'] = pd.to_datetime(weather_df['date'])
# station_meta_df = pd.read_csv('../Data/weather_data_stations/stations_metadata.csv')

# print(weather_df.columns)
# print(weather_df.head())

# print('\n')
# print(station_meta_df.columns)
# print(station_meta_df.head())



#### Restrict to stations in/near Pt Reyes

There are no stations within Pt Reyes, so we create a buffer of 0.08 degrees (about 5 miles) to capture nearby stations in e.g. Bolinas.

In [ ]:
# # Restrict to stations in Point Reyes boundaries

# # Convert your DataFrame to a GeoDataFrame
# station_gdf = gpd.GeoDataFrame(
#     station_meta_df,
#     geometry=gpd.points_from_xy(station_meta_df.longitude, station_meta_df.latitude),
#     crs="EPSG:4326"  # standard Lat/Lon
# )

# # Sync Coordinate Reference Systems
# if pt_reyes_boundary.crs != station_gdf.crs:
#     pt_reyes_boundary = pt_reyes_boundary.to_crs(station_gdf.crs)

# # Filter
# pt_reyes_stations = gpd.clip(station_gdf, pt_reyes_boundary)

# # Create a temporary search area buffered by ~5 miles (roughly 0.08 degrees)
# # This captures stations in nearby towns like Point Reyes Station or Inverness
# # search_area = pt_reyes_boundary.buffer(0.08)
# search_area = pt_reyes_boundary.buffer(0.2)

# # Clip using this expanded area instead
# nearby_stations = gpd.clip(station_gdf, search_area)

# # Check the results
# print(f"Original Stations: {len(station_meta_df)}")
# print(f"Stations Inside Park: {len(pt_reyes_stations)}")
# print(pt_reyes_stations)
# print(f"Nearby Stations: {len(nearby_stations)}")
# print(nearby_stations)

In [ ]:
# # Plot nearby stations along with Pt Reyes boundary

# fig, ax = plt.subplots(figsize=(10, 10))
# pt_reyes_web_mercator.plot(ax=ax, alpha=1.0, facecolor='none', edgecolor='r', linewidth=2)

# # Plot nearby stations
# nearby_stations_web_mercator = nearby_stations.to_crs(epsg=3857)
# nearby_stations_web_mercator.plot(ax=ax, color='blue', markersize=100, alpha=0.5)

# # slightly extend axis boundaries to include nearby stations
# ax.set_xlim(ax.get_xlim()[0] - 30000, ax.get_xlim()[1] + 30000)
# ax.set_ylim(ax.get_ylim()[0] - 30000, ax.get_ylim()[1] + 30000)

# cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
# ax.set_axis_off()

In [ ]:
# nearby_stations_weather = nearby_stations[['station', 'latitude', 'longitude']].merge(weather_df, how='left', on='station')
# nearby_stations_weather.sort_values(by=['station', 'date'], inplace=True)
# nearby_stations_weather.head()
# # nearby_stations_weather.plot(ax=ax, color='green', markersize=100, alpha=0.5)

# print(nearby_stations_weather.PRCP.max())


In [ ]:
# # Plot weather data

# stations = nearby_stations_weather['station'].unique()
# print(stations)
# nstations = len(stations)

# fig, ax = plt.subplots(nstations, 1, figsize=(10, 3*nstations))
# fig.tight_layout()

# for i, station in enumerate(stations):
#     station_data = nearby_stations_weather[nearby_stations_weather['station'] == station]
#     ax[i].plot(station_data['date'], station_data['PRCP'], label=station)
#     ax[i].set_title(station, fontweight='bold')

# # ax.legend()
# plt.show()


### Reanalysis data from Daymet

Many of the weather stations have intermittent or nonexistent data. We'll use historical reanalysis data instead. Daymet has data on a 1km grid.

*Latest data available as of last check (01-01-14) is 12-31-2024. Use Open-Meteo data instead (see below).

In [ ]:
reanalysis_dm_df = pd.read_csv('../Data/weather_data_daymet/data/pt_reyes_weather_history_2010-2024.csv')
reanalysis_dm_df.head()

In [ ]:
import matplotlib.dates as mdates

fig, ax1 = plt.subplots(figsize=(16, 5))

reanalysis_dm_df['date'] = pd.to_datetime(reanalysis_dm_df['date'])

ax1.bar(reanalysis_dm_df['date'], reanalysis_dm_df['prcp_mm'], color='blue', label='Precipitation (mm)', edgecolor='none')
ax1.set_ylabel('Precipitation (mm)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(reanalysis_dm_df['date'], reanalysis_dm_df['tmax_c'], color='red', linewidth=0.2, label='Tmax (°C)')
ax2.plot(reanalysis_dm_df['date'], reanalysis_dm_df['tmin_c'], color='teal', linewidth=0.2, label='Tmin (°C)')
ax2.set_ylabel('Temperature (°C)')
ax2.tick_params(axis='y')

# Optionally add a legend for temperature lines
lines, labels = ax2.get_legend_handles_labels()
ax2.legend(lines, labels, loc='upper right')

# Set x-ticks at each year and rotate labels 45 degrees for better readability
years = reanalysis_dm_df['date'].dt.year.unique()
xticks = [reanalysis_dm_df[reanalysis_dm_df['date'].dt.year == y]['date'].iloc[0] for y in years]
ax1.set_xticks(xticks)
ax1.set_xticklabels(years, rotation=45) #, fontsize=12, color='black')

# Add prominent vertical grid lines at each year to further emphasize years
for xtick in xticks:
    ax1.axvline(x=xtick, color='gray', linestyle='--', linewidth=1, alpha=0.4, zorder=0)

# make horizontal grid lines thinner
ax1.grid(axis='x', which='both', linewidth=0)  # removes default vertical grid lines for ax1
ax1.grid(axis='y', which='both', linewidth=0.5, linestyle=':')
ax2.grid(axis='x', which='both', linewidth=0)  # removes default vertical grid lines for ax2
ax2.grid(axis='y', which='both', linewidth=0.5, linestyle=':')

ax1.set_xlim(pd.Timestamp('2018-01-01'), pd.Timestamp('2020-12-31'))

plt.title('Temp and Precip Reanalysis Data (Daymet), Bear Valley Visitor Center')
fig.tight_layout()
plt.show()

### Open-Meteo Data

Open-Meteo also has historical reanalysis data, but on a coarser 9km grid.

In [ ]:
reanalysis_om_df = pd.read_csv('../Data/weather_data_open-meteo/data/pt_reyes_weather_history_om.csv')
reanalysis_om_df.head()

In [ ]:
import matplotlib.dates as mdates

fig, ax1 = plt.subplots(figsize=(16, 5))

reanalysis_om_df['date'] = pd.to_datetime(reanalysis_om_df['date'])

ax1.bar(reanalysis_om_df['date'], reanalysis_om_df['prcp_mm'], color='blue', label='Precipitation (mm)', edgecolor='none')
ax1.set_ylabel('Precipitation (mm)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(reanalysis_om_df['date'], reanalysis_om_df['tmax_c'], color='red', linewidth=0.2, label='Tmax (°C)')
ax2.plot(reanalysis_om_df['date'], reanalysis_om_df['tmin_c'], color='teal', linewidth=0.2, label='Tmin (°C)')
ax2.set_ylabel('Temperature (°C)')
ax2.tick_params(axis='y')

# Optionally add a legend for temperature lines
lines, labels = ax2.get_legend_handles_labels()
ax2.legend(lines, labels, loc='upper right')

# Set x-ticks at each year and rotate labels 45 degrees for better readability
years = reanalysis_om_df['date'].dt.year.unique()
xticks = [reanalysis_om_df[reanalysis_om_df['date'].dt.year == y]['date'].iloc[0] for y in years]
ax1.set_xticks(xticks)
ax1.set_xticklabels(years, rotation=45) #, fontsize=12, color='black')

# Add prominent vertical grid lines at each year to further emphasize years
for xtick in xticks:
    ax1.axvline(x=xtick, color='gray', linestyle='--', linewidth=1, alpha=0.4, zorder=0)

# make horizontal grid lines thinner
ax1.grid(axis='x', which='both', linewidth=0)  # removes default vertical grid lines for ax1
ax1.grid(axis='y', which='both', linewidth=0.5, linestyle=':')
ax2.grid(axis='x', which='both', linewidth=0)  # removes default vertical grid lines for ax2
ax2.grid(axis='y', which='both', linewidth=0.5, linestyle=':')

# ax1.set_xlim(pd.Timestamp('2018-01-01'), pd.Timestamp('2020-12-31'))

plt.title('Temp and Precip Reanalysis Data (Open-Meteo), Bear Valley Visitor Center')
fig.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go

# Ensure date columns are parsed as datetime
reanalysis_dm_df['date'] = pd.to_datetime(reanalysis_dm_df['date'])
reanalysis_om_df['date'] = pd.to_datetime(reanalysis_om_df['date'])

fig = go.Figure()

fig.add_trace(go.Bar(
    x=reanalysis_dm_df['date'],
    y=reanalysis_dm_df['prcp_mm'],
    name='Precipitation (mm) - Daymet',
    marker_color='blue',
    opacity=1.0
))

fig.add_trace(go.Scatter(
    x=reanalysis_om_df['date'],
    y=reanalysis_om_df['prcp_mm'],
    name='Precipitation (mm) - Open-Meteo',
    mode='markers',
    marker_color='green',
    marker_size=5,
    opacity=1.0
))

fig.update_layout(
    barmode='overlay',
    title='Precip data comparison, Open-Meteo vs Daymet',
    xaxis_title='Date',
    yaxis_title='Precipitation (mm)',
    legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=3, label="3y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    height=400,
    template='plotly_white'
)

fig.show()

In [ ]:
# Plot 7-day moving average of precipitation

# Calculate 7-day moving average
reanalysis_dm_df['prcp_mm_7d_ma'] = reanalysis_dm_df['prcp_mm'].rolling(window=7, min_periods=1).mean()
reanalysis_om_df['prcp_mm_7d_ma'] = reanalysis_om_df['prcp_mm'].rolling(window=7, min_periods=1).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=reanalysis_dm_df['date'],
    y=reanalysis_dm_df['prcp_mm_7d_ma'],
    name='Precipitation (mm) - Daymet',
    mode='lines',
    marker_color='blue',
    marker_size=5,
    opacity=1.0
))

fig.add_trace(go.Scatter(
    x=reanalysis_om_df['date'],
    y=reanalysis_om_df['prcp_mm_7d_ma'],
    name='Precipitation (mm) - Open-Meteo',
    mode='lines',
    marker_color='green',
    marker_size=5,
    opacity=1.0
))

fig.update_layout(
    barmode='overlay',
    title='Precip data comparison (7-day moving average), Open-Meteo vs Daymet',
    xaxis_title='Date',
    yaxis_title='Precipitation (mm)',
    legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=3, label="3y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    height=400,
    template='plotly_white'
)

fig.show()


# **Mushroom Sightings**

In [ ]:
# Load Mushroom Sightings from iNaturalist
print("Loading iNaturalist data...")
inat_df = pd.read_csv('../Data/inaturalist_data/fungi/observations-675943.csv/observations-675943.csv')

In [ ]:
# --- Define Target Species ---
CHOICE_EDIBLES = {
    # 'Chanterelle': ['Cantharellus californicus', 'Cantharellus formosus'],
    'King Bolete': ['Boletus edulis', 'Boletus edulis var. grandedulis'],
    # 'Candy Cap': ['Lactarius rubidus', 'Lactarius rufulus'],
    # 'Black Trumpet': ['Craterellus cornucopioides', 'Craterellus fallax'],
    # 'Hedgehog Mushroom': ['Hydnum repandum']
}

PROXY_SPECIES = {
    # Chanterelles are found with oaks, just like the highly visible Fly Agaric.
    # 'Chanterelle': [
    #     'Amanita muscaria', # Fly Agaric
    #     'Russula'           # Russula (Genus)
    # ],
    
    # King Boletes are pine-associates, sharing habitat with Suillus and Fly Agaric.
    'King Bolete': [
        'Suillus',          # Slippery Jacks (Genus)
        'Amanita muscaria'  # Fly Agaric
    ],

    # Black Trumpets like damp, mossy areas, similar to colorful Waxcaps and Coral Fungi.
    # 'Black Trumpet': [
    #     'Hygrocybe',        # Waxcaps (Genus)
    #     'Ramaria',          # Coral Fungi (Genus)
    #     'Clavaria'          # Coral Fungi (Genus)
    # ],

    # Candy Caps are a type of milk cap; other common milk caps indicate a suitable habitat.
    # 'Candy Cap': [
    #     'Lactarius alnicola' # A common, non-choice Milk Cap
    # ]
}

In [ ]:
# Filter the inaturalist data using scientific names in CHOICE_EDIBLES and PROXY_SPECIES
# (case insensitive, strip whitespace)
edible_sci_names = set(
    species.strip().lower()
    for species_list in CHOICE_EDIBLES.values()
    for species in species_list
)
proxy_sci_names = set(
    species.strip().lower()
    for species_list in PROXY_SPECIES.values()
    for species in species_list
)
all_target_sci_names = edible_sci_names | proxy_sci_names

edibles_df = inat_df[
    inat_df['scientific_name'].str.strip().str.lower().apply(
        lambda sci: any(target in sci for target in all_target_sci_names)
    )
].copy()

In [ ]:
# Define search area as 2km buffer around Pt Reyes boundary
search_area = pt_reyes_boundary.to_crs(epsg=3857).buffer(2000)

# search_area_web_mercator = search_area.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 10))
search_area.plot(ax=ax, alpha=1.0, facecolor='none', edgecolor='r', linewidth=2)
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
# Search for sightings of target species in the search area

# Invert both the CHOICE_EDIBLES and PROXY_SPECIES dictionaries for easy mapping from species name to group
# species_to_group = {}
# for group_dict in [CHOICE_EDIBLES, PROXY_SPECIES]:
#     for group, species_list in group_dict.items():
#         for species in species_list:
#             species_to_group[species] = group

# print(species_to_group)

# Filter for target species and add a 'group' column
# edibles_df = inat_df[inat_df['scientific_name'].isin(species_to_group.keys())].copy()
# edibles_df['group'] = edibles_df['scientific_name'].map(species_to_group)
edibles_df['observed_on'] = pd.to_datetime(edibles_df['observed_on'])

# Add column "proxy_for" to edibles_df, which lists all choice edibles for which the row is a proxy species
EDIBLES_PROXIES_COMBINED = {}
for edible in CHOICE_EDIBLES.keys():
    EDIBLES_PROXIES_COMBINED[edible] = list(CHOICE_EDIBLES.get(edible, []))
    EDIBLES_PROXIES_COMBINED[edible].extend(PROXY_SPECIES.get(edible, []))

# print(EDIBLES_PROXIES_COMBINED)

edibles_df['proxy_for'] = edibles_df['scientific_name'].apply(
    lambda x: [edible for edible in CHOICE_EDIBLES.keys() if any(x.startswith(val) for val in EDIBLES_PROXIES_COMBINED[edible])]
    )

# print(edibles_df[['scientific_name', 'proxy_for']].head())

print(f"Found {len(edibles_df)} choice edible and proxy species sightings.")

# Convert sightings to a GeoDataFrame
edibles_gdf = gpd.GeoDataFrame(
    edibles_df, 
    geometry=gpd.points_from_xy(edibles_df.longitude, edibles_df.latitude),
    crs="EPSG:4326"
)

# Filter sightings to Marin County
search_area_4326 = search_area.to_crs(edibles_gdf.crs)
pt_reyes_edibles_gdf = gpd.clip(edibles_gdf, search_area_4326)

print(f"Found {len(pt_reyes_edibles_gdf)} choice edible and proxy species sightings in Point Reyes region.")

In [ ]:
# Plot all scientific names in the filtered iNaturalist data with at least 20 sightings (most common first)
# import re

order_counts = pt_reyes_edibles_gdf['scientific_name'].value_counts()
order_counts = order_counts[order_counts >= 20]

plot_df = (
    pt_reyes_edibles_gdf[pt_reyes_edibles_gdf['scientific_name'].isin(order_counts.index)]
    .groupby('scientific_name')
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
)

plt.figure(figsize=(12, max(8, 0.5 * len(order_counts))))  # Make figure taller based on number of names

ax = sns.barplot(
    data=plot_df,
    y='scientific_name',
    x='count',
    order=plot_df['scientific_name'],  # ensures identical order
    color='skyblue',
    linewidth=0,
    orient='h',
)

plt.title('Sightings by Scientific Name (Choice Edibles in Bold)')
plt.xlabel('Number of Sightings')

# --- Make y-axis label bold for any choice edible scientific names ---
# Build a flat set of all edible scientific names (lowercased & stripped) from the CHOICE_EDIBLES dict
edibles_sci_names = set(
    species.strip().lower()
    for species_list in CHOICE_EDIBLES.values()
    for species in species_list
)
# Get all tick labels, compare (case-insensitive) if ANY edible name is a substring, and set fontweight
for label in ax.get_yticklabels():
    sci_name = label.get_text().strip().lower()
    if any(edible in sci_name for edible in edibles_sci_names):
        label.set_fontweight('bold')

plt.ylabel('Scientific Name')
plt.tight_layout()
plt.show()

## Sighting Map of Point Reyes region

In [ ]:
# Fix: Preserve original point geometries
fig, ax = plt.subplots(figsize=(15, 15))

# Plot Marin boundary for context
pt_reyes_boundary_web = pt_reyes_boundary.to_crs(epsg=3857)
pt_reyes_boundary_web.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1)

# Use the original pt_reyes_edibles_gdf (which has Point geometries) instead of enriched_gdf
plot_gdf = pt_reyes_edibles_gdf.to_crs(epsg=3857)  # Use original points

# Only keep rows where geometry is Point
points_only = plot_gdf[plot_gdf.geometry.geom_type == "Point"].copy()

if len(points_only) > 0:
    # Extract x/y for scatterplot
    points_only["x"] = points_only.geometry.apply(lambda geom: geom.x)
    points_only["y"] = points_only.geometry.apply(lambda geom: geom.y)
    
    # Get unique groups and colors
    edibles = CHOICE_EDIBLES.keys()
    colors = sns.color_palette("Set2", n_colors=len(edibles))
    # # Get all unique values in the EDIBLES_PROXIES_COMBINED dictionary
    # unique_proxies = set()
    # for v in EDIBLES_PROXIES_COMBINED.values():
    #         unique_proxies.add(v)
    # unique_proxies = list(unique_proxies)
    # colors = sns.color_palette("Set2", n_colors=len(unique_proxies))
    # print(unique_proxies)
    
    # Plot each choice edible group separately with edge colors
    for i, edible in enumerate(edibles):
        # edible_data = points_only[points_only["proxy_for"].apply(lambda proxies: isinstance(proxies, list) and "edible" in proxies)]
        edible_data = points_only[points_only["proxy_for"].apply(lambda proxy_edibles: edible in proxy_edibles)]
        # print(edible_data.head())
        
        # Create darker edge color
        edge_color = tuple([max(c * 0.6, 0) for c in colors[i]])
        
        ax.scatter(
            edible_data["x"], 
            edible_data["y"], 
            c=[colors[i]], 
            s=50, 
            alpha=0.8,
            linewidth=0.8,
            edgecolor=edge_color,
            label=edible
        )

        # ax.scatter(
        #     edible_data["x"], 
        #     edible_data["y"], 
        #     c=[colors[i]], 
        #     s=10, 
        #     alpha=0.9,
        #     linewidth=0.2,
        #     edgecolor=edge_color,
        #     label=edible
        # )
    
    # Add basemap
    cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom=13)
    
    ax.set_title('Porcini/Proxy Sightings in Point Reyes', fontsize=16, fontweight='bold')
    ax.set_axis_off()
    plt.legend(title='Species Group')
    plt.show()
    
    print(f"Plotted {len(points_only)} mushroom sightings")
else:
    print("No points found to plot")

## Sightings over time

In [ ]:
print(pt_reyes_edibles_gdf.columns)
pt_reyes_edibles_gdf.head()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots



# Count reports by date
reports_by_date = pt_reyes_edibles_gdf.groupby('observed_on').size().reset_index(name='count')
# Our weather data starts in 2010 - remove any sightings before then (**iNaturalist was create in 2008, first app was launched in 2011**)
reports_by_date = reports_by_date[reports_by_date['observed_on'] >= pd.Timestamp('2010-01-01')]
print(reports_by_date.head())

# Ensure date columns are parsed as datetime
reports_by_date['observed_on'] = pd.to_datetime(reports_by_date['observed_on'])

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])


# Plot Precipitation (mm) - Open-Meteo
fig.add_trace(go.Scatter(
    x=reanalysis_om_df['date'],
    y=reanalysis_om_df['prcp_mm'],
    name='Precipitation (mm) - Open-Meteo',
    mode='markers',
    marker_color='green',
    marker_size=1,
    opacity=1.0
))

# Plot Precipitation (mm) - Open-Meteo 7-day moving average
fig.add_trace(go.Scatter(
    x=reanalysis_om_df['date'],
    y=reanalysis_om_df['prcp_mm_7d_ma'],
    name='Precipitation (mm) - 7-day moving average',
    mode='lines',
    marker_color='green',
    line_width=1,
    opacity=1.0
))

# Plot mushroom observation counts on a secondary y-axis (right)
fig.add_trace(go.Bar(
    x=reports_by_date['observed_on'],
    y=reports_by_date['count'],
    name='Mushroom Sightings',
    marker_color='blue',
    opacity=1.0
    ),
    secondary_y=True
)


fig.update_layout(
    barmode='overlay',
    title='Mushroom Sightings by Date, with Precipitation',
    xaxis_title='Date',
    yaxis_title='Number of Sightings',
    legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=3, label="3y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    height=400,
    template='plotly_white'
)

fig.show()

#### FFT of sighting counts

Frequency peaks at 1 $y^{-1}$ (yearly) and 1 $w^{-1}$ (weekly). The yearly peak is expected, as porcini are most abundant in the wet season (~Nov-Feb) every year. The weekly peak is due to the fact that more observations occur on the weekend than on weekdays ("observer effort" is higher on weekends).

We can adjust for the weekday/weekend discrepancy by including the day of the week as a training variable and forcing, for example, Saturday as the day of week value when making predicitions (we will want to make similar adjustments for weather conditions - fewer foragers are out on rainy/cold days).

In [ ]:
rpd_copy = reports_by_date.copy()
rpd_copy.set_index('observed_on', inplace=True)
df_filled = rpd_copy.resample('D').asfreq().fillna(0)
print(df_filled.head())

In [ ]:
import scipy.fft

fft = scipy.fft.fft((df_filled['count'] - df_filled['count'].mean()).values)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

ax1.plot(np.arange(len(fft)), np.abs(fft))
ax1.set_title("FFT of Mushroom Sightings")
ax1.set_xlabel(f'Number of Cycles in Full Dataset');
ax1.set_xlim([0,len(fft)/2])

ax2.plot(1/(len(fft)/365.0)*np.arange(len(fft)), np.abs(fft))
ax2.set_title("FFT by yearly freq (zoomed)")
ax2.set_xlabel(f'Frequency ($y^{-1}$)');
ax2.set_xlim([0,7])

ax3.plot(1/(len(fft)/7.0)*np.arange(len(fft)), np.abs(fft))
ax3.set_title("FFT by weekly freq")
ax3.set_xlabel(f'Frequency ($w^{-1}$)');
ax3.set_xlim([0,7/2])

#### Observations are significantly more likely on weekends

In [ ]:
reports_by_date['day_of_week'] = reports_by_date['observed_on'].dt.dayofweek
df_dow = reports_by_date.groupby('day_of_week').count()
print(df_dow.head())
plt.bar(df_dow.index, df_dow['count'])
plt.xticks(ticks=range(7), labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.xlabel('Day of Week')

In [ ]:
# Violin plot of sighting by date during holiday season 
# Are there more sightings on e.g. New Year's Day?

import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data: select sightings count for dates between Nov 15 and Jan 2 (across all years)
# Assumes 'reports_by_date' has 'observed_on' as datetime and 'count'
mask = (
    (reports_by_date['observed_on'].dt.month == 11) & (reports_by_date['observed_on'].dt.day >= 15)
) | (
    (reports_by_date['observed_on'].dt.month == 12)
) | (
    (reports_by_date['observed_on'].dt.month == 1) & (reports_by_date['observed_on'].dt.day <= 2)
)

subset = reports_by_date.loc[mask].copy()
subset['month_day'] = subset['observed_on'].dt.strftime('%m-%d')

plt.figure(figsize=(14,6))
# Ensure the x axis is ordered by date by creating a categorical type with ordered categories
ordered_dates = sorted(subset['month_day'].unique(), key=lambda x: int(x[:2])*100 + int(x[3:]))
subset['month_day'] = pd.Categorical(subset['month_day'], categories=ordered_dates, ordered=True)

sns.violinplot(
    x='month_day',
    y='count',
    data=subset,
    scale='width',
    inner='quartile',
    color='tan'
)
plt.xticks(rotation=90)
plt.xlabel("Date (MM-DD)")
plt.ylabel("Number of Sightings")
plt.title("Distribution of Mushroom Sightings per Date (Holiday Season)")
plt.tight_layout()
plt.show()

## Investigate features

In [ ]:
model_data = reports_by_date.merge(reanalysis_om_df, left_on='observed_on', right_on='date', how='outer')
model_data['count'] = model_data['count'].fillna(0)
model_data.head(20)


In [ ]:

model_data['tmax_c'].plot.hist(bins=15, weights=model_data['count'], edgecolor='black')
plt.xlabel('Max Temp (C)')
plt.show()

model_data['tmin_c'].plot.hist(bins=15, weights=model_data['count'], edgecolor='black')
plt.xlabel('Min Temp (C)')
plt.show()

model_data['prcp_mm'].plot.hist(bins=15, weights=model_data['count'], edgecolor='black')
plt.xlabel('Precip (mm)')
plt.show()

In [ ]:
model_data['yday'].plot.hist(bins=48, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('Day of Year')
plt.show()

In [ ]:
# 14-day precipitation
model_data['14_day_prcp_mm'] = model_data['prcp_mm'].rolling(window=14, closed='left').sum()
# plt.plot(model_data['14_day_prcp_mm'], model_data['count'], 'o', markersize=1)
# plt.show()

# model_data['14_day_prcp_mm_bucket'] = pd.cut(model_data[model_data['count'] < 40]['14_day_prcp_mm'], bins=5, labels=['low', 'medium', 'high', 'very high', 'extreme'])
# binned_plot = model_data.groupby('14_day_prcp_mm_bucket')['count'].mean()
# plt.bar(binned_plot.index, binned_plot.values)
# plt.xlabel('14-day Precipitation Bin')
# plt.ylabel('Sighting Count')
# plt.show()

model_data['14_day_prcp_mm'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('14-day Precipitation (mm)')
plt.show()


In [ ]:
model_data['30_day_prcp_mm'] = model_data['prcp_mm'].rolling(window=30, closed='left').sum()
# plt.plot(model_data['30_day_prcp_mm'], model_data['count'], 'o', markersize=1)
# plt.show()

# model_data['30_day_prcp_mm_bin'] = pd.cut(model_data[model_data['count'] < 40]['30_day_prcp_mm'], bins=5, labels=['low', 'medium', 'high', 'very high', 'extreme'])
# binned_plot = model_data.groupby('30_day_prcp_mm_bin')['count'].mean()
# plt.bar(binned_plot.index, binned_plot.values)
# plt.xlabel('30-day Precipitation Bin')
# plt.ylabel('Sighting Count')
# plt.show()

model_data['30_day_prcp_mm'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('30-day Precipitation (mm)')
plt.show()


In [ ]:
model_data['60_day_prcp_mm'] = model_data['prcp_mm'].rolling(window=60, closed='left').sum()
# plt.plot(model_data['60_day_prcp_mm'], model_data['count'], 'o', markersize=1)
# plt.show()

# model_data['60_day_prcp_mm_bin'] = pd.cut(model_data[model_data['count'] < 40]['60_day_prcp_mm'], bins=5, labels=['low', 'medium', 'high', 'very high', 'extreme'])
# binned_plot = model_data.groupby('60_day_prcp_mm_bin')['count'].mean()
# plt.bar(binned_plot.index, binned_plot.values)
# plt.xlabel('60-day Precipitation Bin')
# plt.ylabel('Sighting Count')
# plt.show()

model_data['60_day_prcp_mm'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('60-day Precipitation (mm)')
plt.show()

In [ ]:
# Compute days since last precip >= 1mm for each day
last_precip_idx = None
days_since_last_precip = []
for idx, row in model_data.iterrows():
    # Counts days since precip >= 1mm 
    # ***before the current day - we want to avoid effect of poor observer effort on rainy days
    if last_precip_idx is None:
        days_since_last_precip.append(None)
    else:
        days_since_last_precip.append(idx - last_precip_idx)

    if row['prcp_mm'] >= 1:
        last_precip_idx = idx
        

model_data['days_since_last_precip_over_1mm'] = days_since_last_precip

# plt.plot(model_data['days_since_last_precip_over_1mm'], model_data['count'], 'o', markersize=1)
# plt.xlabel('Days Since Last Precip >= 1mm')
# plt.ylabel('Sighting Count')
# # plt.xlim(0,10)
# plt.show()

model_data['days_since_last_precip_over_1mm'].plot.hist(bins=40, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('Days Since Last Precip >= 1mm')
plt.show()


In [ ]:
# Compute days since last precip >= 3mm for each day
last_precip_idx = None
days_since_last_precip = []
for idx, row in model_data.iterrows():
    # Counts days since precip >= 1mm 
    # ***before the current day - we want to avoid effect of poor observer effort on rainy days
    if last_precip_idx is None:
        days_since_last_precip.append(None)
    else:
        days_since_last_precip.append(idx - last_precip_idx)

    if row['prcp_mm'] >= 3:
        last_precip_idx = idx

model_data['days_since_last_precip_over_3mm'] = days_since_last_precip

# plt.plot(model_data['days_since_last_precip_over_3mm'], model_data['count'], 'o', markersize=1)
# plt.xlabel('Days Since Last Precip >= 3mm')
# plt.ylabel('Sighting Count')
# plt.show()

model_data['days_since_last_precip_over_3mm'].plot.hist(bins=40, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('Days Since Last Precip >= 3mm')
plt.show()

In [ ]:
# Compute days since last precip >= 5mm for each day
last_precip_idx = None
days_since_last_precip = []
for idx, row in model_data.iterrows():
    # Counts days since precip >= 5mm 
    # ***before the current day - we want to avoid effect of poor observer effort on rainy days
    if last_precip_idx is None:
        days_since_last_precip.append(None)
    else:
        days_since_last_precip.append(idx - last_precip_idx)

    if row['prcp_mm'] >= 5:
        last_precip_idx = idx

model_data['days_since_last_precip_over_5mm'] = days_since_last_precip

# plt.plot(model_data['days_since_last_precip_over_5mm'], model_data['count'], 'o', markersize=1)
# plt.xlabel('Days Since Last Precip >= 5mm')
# plt.ylabel('Sighting Count')
# plt.show()

model_data['days_since_last_precip_over_5mm'].plot.hist(bins=40, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('Days Since Last Precip >= 5mm')
plt.show()

In [ ]:
model_data['7day_tmax_c'] = model_data['tmax_c'].rolling(window=7, closed='left').mean()
# plt.plot(model_data['7day_tmax_c'], model_data['count'], 'o', markersize=1)
# plt.xlabel('7-day Moving Average of Max Temp (C)')
# plt.ylabel('Sighting Count')
# plt.show()
model_data['7day_tmax_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('7-day Moving Average of Max Temp (C)')
plt.show()

model_data['7day_tmin_c'] = model_data['tmin_c'].rolling(window=7, closed='left').mean()
# plt.plot(model_data['7day_tmin_c'], model_data['count'], 'o', markersize=1)
# plt.xlabel('7-day Moving Average of Min Temp (C)')
# plt.ylabel('Sighting Count')
# plt.show()
model_data['7day_tmin_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('7-day Moving Average of Min Temp (C)')
plt.show()



In [ ]:
model_data['14day_tmax_c'] = model_data['tmax_c'].rolling(window=14, closed='left').mean()
# plt.plot(model_data['14day_tmax_c'], model_data['count'], 'o', markersize=1)
# plt.xlabel('14-day Moving Average of Max Temp (C)')
# plt.ylabel('Sighting Count')
# plt.show()
model_data['14day_tmax_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('14-day Moving Average of Max Temp (C)')
plt.show()

model_data['14day_tmin_c'] = model_data['tmin_c'].rolling(window=14, closed='left').mean()
# plt.plot(model_data['14day_tmin_c'], model_data['count'], 'o', markersize=1)
# plt.xlabel('14-day Moving Average of Min Temp (C)')
# plt.ylabel('Sighting Count')
# plt.show()
model_data['14day_tmin_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('14-day Moving Average of Min Temp (C)')
plt.show()

model_data['14-7_tmax_c'] = model_data['14day_tmax_c']-model_data['7day_tmax_c']
# plt.plot(model_data['14-7_tmax_c'], model_data['count'], 'o', markersize=1)
# plt.xlabel('14-day Moving Average of Max Temp (C) - 7-day Moving Average of Max Temp (C)')
# plt.ylabel('Sighting Count')
# plt.show()
model_data['14-7_tmax_c'].plot.hist(bins=10, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('14-day Moving Average of Max Temp (C) - 7-day Moving Average of Max Temp (C)')
plt.show()

model_data['14-7_tmin_c'] = model_data['14day_tmin_c']-model_data['7day_tmin_c']
# plt.plot(model_data['14-7_tmin_c'], model_data['count'], 'o', markersize=1)
# plt.xlabel('14-day Moving Average of Min Temp (C) - 7-day Moving Average of Min Temp (C)')
# plt.ylabel('Sighting Count')
# plt.show()
model_data['14-7_tmin_c'].plot.hist(bins=10, weights=model_data['count']) #, edgecolor='black')
plt.xlabel('14-day Moving Average of Min Temp (C) - 7-day Moving Average of Min Temp (C)')
plt.show()



In [ ]:
model_data['dayofweek'] = model_data['date'].dt.dayofweek

model_data['is_weekend'] = model_data['dayofweek'].isin([5,6])
model_data['is_weekend'].value_counts()

In [ ]:
print(model_data.head())

In [ ]:
# Filter to 2016-2026 (more iNaturalist data, end on an even year for time series split)
start_year = 2016
end_year = 2026

start_date = f'{start_year}-01-01'
end_date = f'{end_year-1}-12-31'

model_data_filtered = model_data[(model_data['date'] >= '2016-01-01') & (model_data['date'] <= '2025-12-31')].copy()
# # Remove the last row if it has nan values
# if model_data_filtered.iloc[-1].isna().any():
#     model_data_filtered = model_data_filtered.iloc[:-1]

# print(model_data_filtered.head())
# print(model_data_filtered[model_data_filtered.isna().any(axis=1)])

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Define a time series split for all cross-validation
test_years = 2 # Leave a couple years for testing
cv = TimeSeriesSplit(n_splits=(end_year-start_year-test_years))

## Old model approaches

In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error, r2_score
# from sklearn.preprocessing import MinMaxScaler
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.linear_model import Ridge
# from sklearn.model_selection import GridSearchCV

# X = model_data_filtered[[
#     # observer effort features
#     'tmax_c', 'tmin_c', 'prcp_mm', 'is_weekend', #'dayofweek',
#     # precip features
#     '14_day_prcp_mm', '30_day_prcp_mm', '60_day_prcp_mm', 
#     'days_since_last_precip_over_1mm', 'days_since_last_precip_over_3mm', 'days_since_last_precip_over_5mm',
#     # temperature features
#     '7day_tmax_c', '7day_tmin_c', '14day_tmax_c', '14day_tmin_c', '14-7_tmax_c', '14-7_tmin_c',
#     # time features (exclude - mushrooms don't know the date)
#     # 'yday'
#     ]].copy()
# X['ones'] = 1
# y = model_data_filtered['count'].copy()

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=(test_years/(end_year-start_year)), shuffle=False)

# pipeline = Pipeline([
#     ('scaler', MinMaxScaler((-1,1))),
#     ('polynomial_trans', PolynomialFeatures()),
#     ('model', Ridge())
# ])

# search = GridSearchCV(
#     pipeline, 
#     param_grid={
#         'polynomial_trans__degree': [1, 2, 4], #, 8, 16, 20],
#         'model__alpha': [0.1, 1, 10, 100, 1000, 10000]
#         }, 
#     cv=cv)

# search.fit(X_train, y_train)

# print("Best parameters found: ", search.best_params_)
# print("Best cross-validation score: ", search.best_score_)
# print("Best model: ", search.best_estimator_)
# print("Best model_coefs: ", search.best_estimator_.named_steps['model'].coef_)

# y_pred = search.predict(X_test)

# mse = mean_squared_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# print(f"MSE: {mse}")
# print(f"R2: {r2}")

# plt.scatter(y_test, y_pred)
# plt.xlabel('Actual Count')
# plt.ylabel('Predicted Count')
# plt.title('Actual vs Predicted Counts')
# plt.show()





In [ ]:
# # Define features for modeling using trees
# X = model_data_filtered[[
#     # observer effort features
#     'tmax_c', 'tmin_c', 'prcp_mm', 'is_weekend', #'dayofweek',
#     # precip features
#     '14_day_prcp_mm', '30_day_prcp_mm', '60_day_prcp_mm', 
#     'days_since_last_precip_over_1mm', 'days_since_last_precip_over_3mm', 'days_since_last_precip_over_5mm',
#     # temperature features
#     '7day_tmax_c', '7day_tmin_c', '14day_tmax_c', '14day_tmin_c', '14-7_tmax_c', '14-7_tmin_c',
#     # time features (exclude - mushrooms don't know the date)
#     # 'yday'
#     ]].copy()
# y = model_data_filtered['count'].copy()

# # Test train split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=(test_years/(end_year-start_year)), shuffle=False)

In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error, r2_score
# from sklearn.preprocessing import MinMaxScaler
# from sklearn.pipeline import Pipeline
# from sklearn.tree import DecisionTreeRegressor
# from sklearn.model_selection import GridSearchCV

# pipeline = Pipeline([
#     # ('scaler', MinMaxScaler((-1,1))),
#     ('model', DecisionTreeRegressor())
# ])

# search = GridSearchCV(
#     pipeline, 
#     param_grid={
#         'model__max_depth': [2, 3, 5, 10, 20],
#         'model__min_samples_split': [2, 5, 10],
#         'model__min_samples_leaf': [1, 2, 5]
#     }, 
#     cv=cv,
#     # scoring='explained_variance'
#     )

# search.fit(X_train, y_train)

# print("Best parameters found: ", search.best_params_)
# print("Best cross-validation score: ", search.best_score_)
# print("Best model: ", search.best_estimator_)
# # print("Best model_coefs: ", search.best_estimator_.named_steps['model'].coef_)

# y_pred = search.predict(X_test)

# mse = mean_squared_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# print(f"MSE: {mse}")
# print(f"R2: {r2}")

# plt.scatter(y_test, y_pred)
# plt.xlabel('Actual Count')
# plt.ylabel('Predicted Count')
# plt.title('Actual vs Predicted Counts')
# plt.show()



In [ ]:
# print(X.columns)

# import os
# # Homebrew on Apple Silicon uses /opt/homebrew/bin; Intel Macs often use /usr/local/bin
# os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')
# # Now import and use graphviz
# import graphviz

# from sklearn.tree import export_graphviz

# graphviz.Source(export_graphviz(search.best_estimator_.named_steps['model'], feature_names=X.columns))






In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error, r2_score
# from sklearn.preprocessing import MinMaxScaler
# from sklearn.pipeline import Pipeline
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import GridSearchCV



# pipeline = Pipeline([
#     # ('scaler', MinMaxScaler((-1,1))),
#     ('model', RandomForestRegressor())
# ])

# search = GridSearchCV(
#     pipeline, 
#     param_grid={
#         'model__n_estimators': [66, 100, 200],
#         'model__max_depth': [2, 5, 10],
#         'model__min_samples_split': [5, 10],
#         'model__min_samples_leaf': [2, 5]
#     }, 
#     cv=cv,
#     scoring='neg_mean_absolute_error'
#     )

# search.fit(X_train, y_train)

# print("Best parameters found: ", search.best_params_)
# print("Best cross-validation score: ", search.best_score_)
# print("Best model: ", search.best_estimator_)
# # print("Best model_coefs: ", search.best_estimator_.named_steps['model'].coef_)

# y_pred = search.predict(X_test)

# mse = mean_squared_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# print(f"MSE: {mse}")
# print(f"R2: {r2}")

# plt.scatter(y_test, y_pred)
# plt.xlabel('Actual Count')
# plt.ylabel('Predicted Count')
# plt.title('Actual vs Predicted Counts')
# plt.show()



In [ ]:
# columns = X.columns
# # Get the best RandomForestRegressor from the pipeline
# best_rf = search.best_estimator_.named_steps['model']
# print(best_rf.feature_importances_)

# # Plot the feature importances
# plt.bar(columns, best_rf.feature_importances_)
# plt.xlabel('Features')
# plt.ylabel('Importance')
# plt.xticks(rotation=90)
# plt.show()

In [ ]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# # Create figure with secondary y-axis
# fig = make_subplots(specs=[[{"secondary_y": True}]])


# # Plot Precipitation (mm) - Open-Meteo
# fig.add_trace(go.Bar(
#     x=model_data_filtered['date'],
#     y=model_data_filtered['prcp_mm'],
#     name='Precipitation (mm) - Open-Meteo',
#     # mode='markers',
#     # marker_color='blue',
#     # marker_size=1,
#     opacity=1.0
# ))

# # # Plot Precipitation (mm) - Open-Meteo 7-day moving average
# # fig.add_trace(go.Scatter(
# #     x=model_data_filtered['date'],
# #     y=model_data_filtered['prcp_mm_7d_ma'],
# #     name='Precipitation (mm) - 7-day moving average',
# #     mode='lines',
# #     marker_color='green',
# #     line_width=1,
# #     opacity=1.0
# # ))

# # Plot mushroom observation counts on a secondary y-axis (right)
# # model_data_filtered = model_data_filtered.sort_values(by='observed_on')
# fig.add_trace(go.Scatter(
#     x=model_data_filtered['date'],
#     y=model_data_filtered['count'],
#     name='Mushroom Sightings',
#     mode='markers',
#     marker_color='brown',
#     marker_size=5,
#     marker_symbol='circle',
#     opacity=1.0
#     ),
#     secondary_y=True
# )

# # # Plot Precipitation (mm) - Open-Meteo 7-day moving average
# # fig.add_trace(go.Scatter(
# #     x=reanalysis_om_df['date'],
# #     y=reanalysis_om_df['prcp_mm_7d_ma'],
# #     name='Precipitation (mm) - 7-day moving average',
# #     mode='lines',
# #     marker_color='green',
# #     line_width=1,
# #     opacity=1.0
# # ))

# # Plot predictedmushroom observation counts on a secondary y-axis (right)
# predX = model_data_filtered.copy()
# predX['predicted_count'] = search.predict(predX[[
#     # observer effort features
#     'tmax_c', 'tmin_c', 'prcp_mm', 'is_weekend', #'dayofweek',
#     # precip features
#     '14_day_prcp_mm', '30_day_prcp_mm', '60_day_prcp_mm', 
#     'days_since_last_precip_over_1mm', 'days_since_last_precip_over_3mm', 'days_since_last_precip_over_5mm',
#     # temperature features
#     '7day_tmax_c', '7day_tmin_c', '14day_tmax_c', '14day_tmin_c', '14-7_tmax_c', '14-7_tmin_c',
#     # time features (exclude - mushrooms don't know the date)
#     # 'yday'
#     ]])
# predX = predX.sort_values(by='date')
# fig.add_trace(go.Scatter(
#     x=predX['date'],
#     y=predX['predicted_count'],
#     name='Predicted Sightings',
#     mode='lines+markers',
#     marker_color='green',
#     marker_size=4,
#     line_width=1,
#     marker_symbol='circle',
#     opacity=1.0
#     ),
#     secondary_y=True
# )

# # print(predX['observed_on'].min())

# # Plot predicted mushroom observation counts under ideal conditions (weekend, good weather)
# predX_ideal = predX.copy()
# predX_ideal['is_weekend'] = True
# predX_ideal['tmax_c'] = 12
# predX_ideal['tmin_c'] = 8
# predX_ideal['prcp_mm'] = 0
# predX_ideal['predicted_count'] = search.predict(predX_ideal[[
#     # observer effort features
#     'tmax_c', 'tmin_c', 'prcp_mm', 'is_weekend', #'dayofweek',
#     # precip features
#     '14_day_prcp_mm', '30_day_prcp_mm', '60_day_prcp_mm', 
#     'days_since_last_precip_over_1mm', 'days_since_last_precip_over_3mm', 'days_since_last_precip_over_5mm',
#     # temperature features
#     '7day_tmax_c', '7day_tmin_c', '14day_tmax_c', '14day_tmin_c', '14-7_tmax_c', '14-7_tmin_c',
#     # time features (exclude - mushrooms don't know the date)
#     # 'yday'
#     ]])
# predX_ideal = predX_ideal.sort_values(by='date')
# fig.add_trace(go.Scatter(
#     x=predX_ideal['date'],
#     y=predX_ideal['predicted_count'],
#     name='Predicted Sightings (Ideal)',
#     mode='lines+markers',
#     marker_color='blue',
#     marker_size=4,
#     line_width=1,
#     marker_symbol='circle',
#     opacity=1.0
#     ),
#     secondary_y=True
# )

# fig.update_layout(
#     barmode='overlay',
#     title='Mushroom Sightings and Predictions by Date, with Precipitation',
#     xaxis_title='Date',
#     yaxis_title='Number of Sightings',
#     legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
#     xaxis=dict(
#         rangeselector=dict(
#             buttons=list([
#                 dict(count=1, label="1y", step="year", stepmode="backward"),
#                 dict(count=3, label="3y", step="year", stepmode="backward"),
#                 dict(step="all")
#             ])
#         ),
#         rangeslider=dict(
#             visible=True
#         ),
#         type="date"
#     ),
#     height=400,
#     template='plotly_white'
# )

# fig.show()

## Latest modelling approach

#### Try including yday - how much "better" does the model perform?

In [ ]:
# model_data_filtered['days_since_july_1'] =  ((model_data_filtered['yday'] - 182) % 365).astype(int)

# model_data_filtered['days_since_july_1'].describe()

# plt.plot(model_data_filtered['yday'], model_data_filtered['days_since_july_1'])
# plt.show()



In [ ]:
model_data_filtered['sin_time'] = np.sin(2 * np.pi * model_data_filtered['yday'] / 365)
model_data_filtered['cos_time'] = np.cos(2 * np.pi * model_data_filtered['yday'] / 365)

plt.plot(model_data_filtered['yday'], model_data_filtered['sin_time'], 'r-', label='sin_time')
plt.plot(model_data_filtered['yday'], model_data_filtered['cos_time'], 'b-', label='cos_time')
plt.legend()
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

model_features = [
    # observer effort features
    'tmax_c', 'tmin_c', 'prcp_mm', 'is_weekend', #'dayofweek',
    # precip features
    '14_day_prcp_mm', '30_day_prcp_mm', '60_day_prcp_mm', 
    # 'days_since_last_precip_over_1mm', 'days_since_last_precip_over_3mm', 
    'days_since_last_precip_over_5mm',
    # temperature features
    '7day_tmax_c', '7day_tmin_c', '14day_tmax_c', '14day_tmin_c', '14-7_tmax_c', '14-7_tmin_c',
    # time features (exclude - mushrooms don't know the date)
    'sin_time', 'cos_time'
]

X = model_data_filtered[model_features].copy()
# X['ones'] = 1
y = model_data_filtered['count'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=(test_years/(end_year-start_year)), shuffle=False)

In [ ]:
print(max(model_data_filtered['count']))
print(max(y_train))
print(max(y_test))
print(test_years/(end_year-start_year))
print(len(y_test))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    # ('scaler', MinMaxScaler((-1,1))),
    ('model', RandomForestRegressor())
])

search = GridSearchCV(
    pipeline, 
    param_grid={
        'model__n_estimators': [20, 40, 80, 160],
        'model__max_depth': [2, 5, 8],
        'model__min_samples_split': [8, 16],
        'model__min_samples_leaf': [4, 8, 16]
    }, 
    cv=cv,
    scoring='neg_mean_absolute_error'
    )

search.fit(X_train, y_train)

print("Best parameters found: ", search.best_params_)
print("Best cross-validation score: ", search.best_score_)
print("Best model: ", search.best_estimator_)
# print("Best model_coefs: ", search.best_estimator_.named_steps['model'].coef_)

y_pred = search.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse}")
print(f"R2: {r2}")

plt.scatter(y_test, y_pred)
plt.xlabel('Actual Count')
plt.ylabel('Predicted Count')
plt.title('Actual vs Predicted Counts')
plt.show()

In [ ]:
columns = X.columns
# Get the best RandomForestRegressor from the pipeline
best_rf = search.best_estimator_.named_steps['model']
print(best_rf.feature_importances_)

# Plot the feature importances
plt.bar(columns, best_rf.feature_importances_)
plt.xlabel('Features')
plt.ylabel('Importance')
plt.xticks(rotation=90)
plt.show()

In [ ]:
print(y)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])


# Plot Precipitation (mm) - Open-Meteo
fig.add_trace(go.Bar(
    x=model_data_filtered['date'],
    y=model_data_filtered['prcp_mm'],
    name='Precipitation (mm) - Open-Meteo',
    # mode='markers',
    # marker_color='blue',
    # marker_size=1,
    opacity=1.0
))

# # Plot Precipitation (mm) - Open-Meteo 7-day moving average
# fig.add_trace(go.Scatter(
#     x=model_data_filtered['date'],
#     y=model_data_filtered['prcp_mm_7d_ma'],
#     name='Precipitation (mm) - 7-day moving average',
#     mode='lines',
#     marker_color='green',
#     line_width=1,
#     opacity=1.0
# ))

# Plot mushroom observation counts on a secondary y-axis (right)
model_data_filtered = model_data_filtered.sort_values(by='observed_on')
fig.add_trace(go.Scatter(
    x=model_data_filtered['date'],
    y=model_data_filtered['count'],
    name='Mushroom Sightings',
    mode='markers',
    marker_color='brown',
    marker_size=5,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

# # Plot Precipitation (mm) - Open-Meteo 7-day moving average
# fig.add_trace(go.Scatter(
#     x=reanalysis_om_df['date'],
#     y=reanalysis_om_df['prcp_mm_7d_ma'],
#     name='Precipitation (mm) - 7-day moving average',
#     mode='lines',
#     marker_color='green',
#     line_width=1,
#     opacity=1.0
# ))

# Plot predicted mushroom observation counts on a secondary y-axis (right)
predX = model_data_filtered.copy()
predX['predicted_count'] = search.predict(predX[model_features])
predX = predX.sort_values(by='date')
fig.add_trace(go.Scatter(
    x=predX['date'],
    y=predX['predicted_count'],
    name='Predicted Sightings',
    mode='lines+markers',
    marker_color='green',
    marker_size=4,
    line_width=1,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

# print(predX['observed_on'].min())

# Plot predicted mushroom observation counts under ideal conditions (weekend, good weather)
predX_ideal = predX.copy()
predX_ideal['is_weekend'] = True
predX_ideal['tmax_c'] = 12
predX_ideal['tmin_c'] = 8
predX_ideal['prcp_mm'] = 0
predX_ideal['predicted_count'] = search.predict(predX_ideal[model_features])
predX_ideal = predX_ideal.sort_values(by='date')
fig.add_trace(go.Scatter(
    x=predX_ideal['date'],
    y=predX_ideal['predicted_count'],
    name='Predicted Sightings (Ideal)',
    mode='lines+markers',
    marker_color='blue',
    marker_size=4,
    line_width=1,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

fig.update_layout(
    barmode='overlay',
    title='Mushroom Sightings and Predictions by Date, with Precipitation',
    xaxis_title='Date',
    yaxis_title='Number of Sightings',
    legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=3, label="3y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    height=400,
    template='plotly_white'
)

fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(predX['cos_time'], predX['predicted_count'], linewidth=1)
# ax.plot([366-182, 366-182], [0, 8], 'k--', linewidth=1, label='Jan 1st')
ax.set_xlabel('Cosine Time')
ax.set_ylabel('Predicted Count')
ax.set_title('Predicted Count by Cosine Time')
ax.legend()
plt.show()


In [ ]:
# Save the best model
import pickle

# Save the best model to a file
import datetime
current_date = datetime.datetime.now().strftime('%Y-%m-%d')
with open(f'model_{current_date}.pkl', 'wb') as f:
    pickle.dump(search.best_estimator_, f)


In [ ]:
predX[['date'] + model_features].to_csv('predX.csv', index=False)